In [1]:
# ============================================================
# PHARMA EVIDENCE API — Full Pipeline (clean rebuild)
# Harvest → Normalize → Parse Results → Score → Export KO JSON
# ============================================================

import requests
import pandas as pd
import math
import json
from datetime import datetime, timezone
from scipy.stats import norm

# ------------------------------------------------------------
# STEP 1: HARVEST — pull the 8 trials from ClinicalTrials.gov
# ------------------------------------------------------------
BASE_URL = "https://clinicaltrials.gov/api/v2/studies"
params = {
    "query.intr": "adagrasib OR sotorasib OR divarasib",
    "query.cond": "NSCLC",
    "filter.overallStatus": "COMPLETED",
    "pageSize": 100,
    "format": "json"
}

print("Calling ClinicalTrials.gov API...")
response = requests.get(BASE_URL, params=params)

if response.status_code == 200:
    print(f"✅ Success! Status code: {response.status_code}")
    data = response.json()
else:
    raise RuntimeError(f"API call failed: {response.status_code} — {response.text}")

studies = data.get("studies", [])
print(f"Studies returned: {len(studies)}")

rows = []
for study in studies:
    protocol = study.get("protocolSection", {})
    identification = protocol.get("identificationModule", {})
    design = protocol.get("designModule", {})
    sponsor = protocol.get("sponsorCollaboratorsModule", {})
    outcomes = protocol.get("outcomesModule", {})
    interventions = protocol.get("armsInterventionsModule", {})
    has_results = study.get("hasResults", False)

    nct_id = identification.get("nctId", "Unknown")
    title = identification.get("briefTitle", "Unknown")
    phase_list = design.get("phases", [])
    phase = ", ".join(phase_list) if phase_list else "Not specified"
    enrollment = design.get("enrollmentInfo", {}).get("count", None)
    lead_sponsor = sponsor.get("leadSponsor", {}).get("name", "Unknown")
    intervention_names = [i.get("name", "") for i in interventions.get("interventions", [])]
    intervention_str = "; ".join(intervention_names)
    primary_outcomes = outcomes.get("primaryOutcomes", [])
    primary_outcome_str = "; ".join([po.get("measure", "") for po in primary_outcomes])

    rows.append({
        "NCTId": nct_id, "BriefTitle": title, "Phase": phase,
        "EnrollmentCount": enrollment, "LeadSponsor": lead_sponsor,
        "Interventions": intervention_str, "PrimaryOutcomes": primary_outcome_str,
        "HasResultsPosted": has_results
    })

df = pd.DataFrame(rows)
print(f"\n✅ Harvested {len(df)} trials")
print(df[["NCTId", "BriefTitle", "EnrollmentCount", "HasResultsPosted"]])


# ------------------------------------------------------------
# STEP 2: NORMALIZE — manual drug code lookup (v0, hand-maintained)
# ------------------------------------------------------------
DRUG_CODE_MAP = {
    "MRTX849": "Adagrasib", "ADAGRASIB": "Adagrasib",
    "AMG 510": "Sotorasib", "AMG510": "Sotorasib", "SOTORASIB": "Sotorasib",
    "GDC-6036": "Divarasib", "DIVARASIB": "Divarasib",
    "TNO155": "TNO155", "DURVALUMAB": "Durvalumab"
}

def normalize_drug_string(raw_string):
    if pd.isna(raw_string) or raw_string == "":
        return ""
    individual_drugs = raw_string.split("; ")
    normalized_list = []
    for drug in individual_drugs:
        drug_clean = drug.strip()
        lookup_key = drug_clean.upper()
        if lookup_key in DRUG_CODE_MAP:
            normalized_list.append(DRUG_CODE_MAP[lookup_key])
        else:
            normalized_list.append(drug_clean + " [UNMAPPED]")
    unique_normalized = list(dict.fromkeys(normalized_list))
    return "; ".join(unique_normalized)

df["Normalized_Drugs"] = df["Interventions"].apply(normalize_drug_string)
print("\n✅ Normalization complete")
print(df[["NCTId", "Interventions", "Normalized_Drugs"]])


# ------------------------------------------------------------
# STEP 3: GLOBAL CONFIG — clinical assumptions (pending pharmacologist review)
# ------------------------------------------------------------
DEFAULT_TARGET_HAZARD_RATIO = 0.75
ALPHA = 0.05
POWER = 0.80
ASSUMED_EVENT_RATE = 0.85
PHARMACOLOGIST_REVIEW_REQUIRED = True

Z_ALPHA = norm.ppf(1 - ALPHA / 2)
Z_BETA = norm.ppf(POWER)

def calculate_required_n(hazard_ratio=DEFAULT_TARGET_HAZARD_RATIO,
                          event_rate=ASSUMED_EVENT_RATE,
                          z_alpha=Z_ALPHA, z_beta=Z_BETA):
    required_events = 4 * ((z_alpha + z_beta) ** 2) / (math.log(hazard_ratio) ** 2)
    return math.ceil(required_events / event_rate)

REQUIRED_N = calculate_required_n()
print(f"\nUsing HR={DEFAULT_TARGET_HAZARD_RATIO}, alpha={ALPHA}, power={POWER}")
print(f"--> Required enrollment per this assumption: {REQUIRED_N} patients")


# ------------------------------------------------------------
# STEP 4: RESULTS SECTION PARSER — real p-values, CIs, outcome values
# ------------------------------------------------------------
def fetch_full_study(nct_id):
    url = f"https://clinicaltrials.gov/api/v2/studies/{nct_id}"
    response = requests.get(url, params={"format": "json"})
    if response.status_code == 200:
        return response.json()
    print(f"⚠️ Failed to fetch {nct_id}: status {response.status_code}")
    return None

def parse_results_section(study_json):
    extracted = {
        "PrimaryEndpoint_Description": None, "PrimaryEndpoint_InterventionValue": None,
        "PrimaryEndpoint_ComparatorValue": None, "PrimaryEndpoint_Unit": None,
        "PrimaryEndpoint_PValue": None, "PrimaryEndpoint_CI_Lower": None,
        "PrimaryEndpoint_CI_Upper": None, "extraction_confidence": "not_extracted"
    }
    results_section = study_json.get("resultsSection")
    if not results_section:
        return extracted

    outcome_module = results_section.get("outcomeMeasuresModule", {})
    outcome_measures = outcome_module.get("outcomeMeasures", [])
    primary_outcomes = [om for om in outcome_measures if om.get("type") == "PRIMARY"]
    if not primary_outcomes:
        return extracted

    primary = primary_outcomes[0]
    extracted["PrimaryEndpoint_Description"] = primary.get("title")
    extracted["PrimaryEndpoint_Unit"] = primary.get("unitOfMeasure")

    classes = primary.get("classes", [])
    if classes:
        categories = classes[0].get("categories", [])
        if categories:
            measurements = categories[0].get("measurements", [])
            if len(measurements) >= 1:
                extracted["PrimaryEndpoint_InterventionValue"] = measurements[0].get("value")
            if len(measurements) >= 2:
                extracted["PrimaryEndpoint_ComparatorValue"] = measurements[1].get("value")

    analyses = primary.get("analyses", [])
    if analyses:
        analysis = analyses[0]
        p_val_raw = analysis.get("pValue")
        try:
            extracted["PrimaryEndpoint_PValue"] = float(p_val_raw) if p_val_raw else None
        except (ValueError, TypeError):
            extracted["PrimaryEndpoint_PValue"] = None
        try:
            ci_lower = analysis.get("ciLowerLimit")
            ci_upper = analysis.get("ciUpperLimit")
            extracted["PrimaryEndpoint_CI_Lower"] = float(ci_lower) if ci_lower else None
            extracted["PrimaryEndpoint_CI_Upper"] = float(ci_upper) if ci_upper else None
        except (ValueError, TypeError):
            pass

    if extracted["PrimaryEndpoint_PValue"] is not None:
        extracted["extraction_confidence"] = "high"
    elif extracted["PrimaryEndpoint_Description"] is not None:
        extracted["extraction_confidence"] = "medium"
    else:
        extracted["extraction_confidence"] = "low"
    return extracted

results_rows = []
for nct_id in df["NCTId"]:
    print(f"Fetching full record for {nct_id}...")
    study_json = fetch_full_study(nct_id)
    parsed = parse_results_section(study_json) if study_json else parse_results_section({})
    parsed["NCTId"] = nct_id
    results_rows.append(parsed)

df_results = pd.DataFrame(results_rows)
df = df.merge(df_results, on="NCTId", how="left")
print("\n✅ Results parsing complete")
print(df[["NCTId", "PrimaryEndpoint_Description", "PrimaryEndpoint_PValue", "extraction_confidence"]])


# ------------------------------------------------------------
# STEP 5: TRUST SCORING — real power check + p-value + CI + results checks
# ------------------------------------------------------------
def statistical_validity_score(row, required_n=REQUIRED_N):
    score = 100
    notes = []

    enrollment = row.get("EnrollmentCount")
    if pd.isna(enrollment):
        score -= 25
        notes.append("Enrollment count missing — cannot verify power")
    elif enrollment < required_n:
        score -= 25
        notes.append(f"Underpowered: enrolled {enrollment} vs. required {required_n} (assumes HR={DEFAULT_TARGET_HAZARD_RATIO})")
    else:
        notes.append(f"Adequately powered: enrolled {enrollment} vs. required {required_n} (assumes HR={DEFAULT_TARGET_HAZARD_RATIO})")

    p_val = row.get("PrimaryEndpoint_PValue")
    if p_val is not None and not pd.isna(p_val):
        if 0.04 < p_val <= 0.05:
            score -= 10
            notes.append(f"Borderline p-value ({p_val}) — elevated p-hacking risk")
    else:
        notes.append("No p-value extracted — cannot assess")

    ci_lower = row.get("PrimaryEndpoint_CI_Lower")
    if ci_lower is None or pd.isna(ci_lower):
        score -= 15
        notes.append("No confidence interval reported")

    if row.get("HasResultsPosted") == False:
        score -= 50
        notes.append("No results posted — cannot verify any claims")

    return max(0, score), notes

score_results = df.apply(statistical_validity_score, axis=1)
df["TrustScore"] = score_results.apply(lambda x: x[0])
df["TrustScore_Notes"] = score_results.apply(lambda x: x[1])
print("\n✅ Trust scoring complete")
print(df[["NCTId", "TrustScore"]])


# ------------------------------------------------------------
# STEP 6: BUILD KNOWLEDGE OBJECTS — real data only, nulls where unextracted
# ------------------------------------------------------------
def build_knowledge_object(row, index):
    return {
        "ko_id": f"KO-2026-NSCLC-{str(index).zfill(5)}",
        "version": "0.1",
        "created_at": datetime.now(timezone.utc).isoformat(),
        "status": "draft_pending_pharmacologist_review",
        "source": {"primary_source": "clinicaltrials_gov", "nct_id": row["NCTId"]},
        "study_metadata": {
            "title": row["BriefTitle"], "phase": row["Phase"],
            "enrollment_actual": None if pd.isna(row["EnrollmentCount"]) else int(row["EnrollmentCount"]),
            "lead_sponsor": row["LeadSponsor"]
        },
        "entities": {
            "intervention": {
                "normalized_drugs": row["Normalized_Drugs"],
                "confidence": "medium" if "[UNMAPPED]" not in row["Normalized_Drugs"] else "low"
            }
        },
        "pico": {
            "population": {
                "sample_size": None if pd.isna(row["EnrollmentCount"]) else int(row["EnrollmentCount"]),
                "description": None,
                "confidence": "not_extracted"
            },
            "outcomes": [{
                "type": "primary",
                "description": row.get("PrimaryEndpoint_Description"),
                "confidence": row.get("extraction_confidence")
            }]
        },
        "results": {
            "primary_endpoint": {
                "outcome": row.get("PrimaryEndpoint_Description"),
                "intervention_value": row.get("PrimaryEndpoint_InterventionValue"),
                "comparator_value": row.get("PrimaryEndpoint_ComparatorValue"),
                "unit": row.get("PrimaryEndpoint_Unit"),
                "p_value": row.get("PrimaryEndpoint_PValue"),
                "confidence_interval_95": [row.get("PrimaryEndpoint_CI_Lower"), row.get("PrimaryEndpoint_CI_Upper")]
                    if row.get("PrimaryEndpoint_CI_Lower") is not None else None
            },
            "results_posted": bool(row["HasResultsPosted"])
        },
        "trust_signals": {
            "statistical_validity_score": int(row["TrustScore"]),
            "statistical_validity_notes": row["TrustScore_Notes"],
            "statistical_validity_assumptions": {
                "target_hazard_ratio": DEFAULT_TARGET_HAZARD_RATIO,
                "assumed_event_rate": ASSUMED_EVENT_RATE,
                "alpha": ALPHA, "power": POWER,
                "pharmacologist_reviewed": not PHARMACOLOGIST_REVIEW_REQUIRED
            },
            "replication_confidence_score": None,
            "coi_flags": None,
            "retraction_status": None
        },
        "agent_rationale": None
    }


# ------------------------------------------------------------
# STEP 7: EXPORT — the ONLY write to kras_g12c_knowledge_objects.json
# ------------------------------------------------------------
print(f"\nRows currently in df: {len(df)}")
assert len(df) == 8, f"Expected 8 trials, found {len(df)} — stop and check df before exporting"

knowledge_objects = [build_knowledge_object(row, i+1) for i, row in df.iterrows()]
assert len(knowledge_objects) == len(df), "Mismatch between df rows and KO objects built"
print(f"✅ Built {len(knowledge_objects)} knowledge objects")

with open("kras_g12c_knowledge_objects.json", "w") as f:
    json.dump(knowledge_objects, f, indent=2, default=str)

with open("kras_g12c_knowledge_objects.json", "r") as f:
    check = json.load(f)
print(f"✅ Verified on disk: {len(check)} objects in kras_g12c_knowledge_objects.json")
assert len(check) == 8, "File on disk doesn't match expected count — do not upload yet"

print("\n--- Confirming all 8 are distinct trials ---")
for ko in knowledge_objects:
    print(f"  {ko['ko_id']}  →  {ko['source']['nct_id']}")

Calling ClinicalTrials.gov API...
✅ Success! Status code: 200
Studies returned: 8

✅ Harvested 8 trials
         NCTId                                         BriefTitle  \
0  NCT04720976  JAB-3312 Based Combination Therapy in Adult Pa...   
1  NCT04330664  Adagrasib in Combination With TNO155 in Patien...   
2  NCT06333678  A Study Comparing Sotorasib With Durvalumab in...   
3  NCT05054725  Combination Study of RMC-4630 and Sotorasib fo...   
4  NCT06314763            Rivaroxaban Sotorasib Interaction Study   
5  NCT04959981  A Study of Anti-Cancer Therapies Targeting the...   
6  NCT07143513  Real World Evaluation of Sotorasib Among Chine...   
7  NCT04933695  A Study of Sotorasib (AMG 510) in Participants...   

   EnrollmentCount  HasResultsPosted  
0               58             False  
1               86             False  
2               10             False  
3               47              True  
4               21             False  
5               24             False  
6